# Requirements
! pip freeze | grep "tensorflow"
tensorflow==2.9.1
tensorflow-estimator==2.9.0
tensorflow-io-gcs-filesystem==0.26.0

! pip freeze | grep "keras"     
keras==2.9.0
keras-bert==0.89.0
keras-embed-sim==0.10.0
keras-layer-normalization==0.16.0
keras-multi-head==0.29.0
keras-pos-embd==0.13.0
keras-position-wise-feed-forward==0.8.0
keras-self-attention==0.51.0
keras-transformer==0.40.0
seqtag-keras==1.0.6

# Load Teachable Machine Model and Run Forward Pass Inference

In [1]:
from keras.models import load_model
from PIL import Image, ImageOps
import numpy as np

# Load the model
model = load_model('maskClassifier/keras_model.h5')

model.summary()

2025-04-08 18:28:36.743375: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_1 (Sequential)   (None, 1280)              410208    
                                                                 
 sequential_3 (Sequential)   (None, 2)                 128300    
                                                                 
Total params: 538,508
Trainable params: 524,428
Non-trainable params: 14,080
_________________________________________________________________


In [3]:
# Create the array of the right shape to feed into the keras model
# The 'length' or number of images you can put into the array is
# determined by the first position in the shape tuple, in this case 1.
data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)

# Replace this with the path to your image
image = Image.open('dataset/with_mask/0-with-mask.jpg')

#resize the image to a 224x224 with the same strategy as in TM2:
#resizing the image to be at least 224x224 and then cropping from the center
size = (224, 224)
image = ImageOps.fit(image, size) #, Image.ANTIALIAS)

#turn the image into a numpy array
image_array = np.asarray(image)
# Normalize the image
normalized_image_array = (image_array.astype(np.float32) / 127.0) - 1
# Load the image into the array
data[0] = normalized_image_array

# run the inference
prediction = model.predict(data)
print(100*prediction)

1/1 [==============================] - 1s 1s/step
[[  0. 100.]]


# TRANSFER LEARNING - Making Embeddings

# Lets try and use ImageNet based InceptionV3 as a feature extractor to see whats in the mind of the neural network that can classify 1000 different classes!

In [4]:
import tensorflow as tf
from tensorflow import keras 
from tensorflow.keras import Model
from tensorflow.keras.applications.resnet50 import ResNet50
import cv2
from tensorflow.keras.preprocessing import image
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions
import matplotlib.pyplot as plt
from tensorflow.keras.layers import GlobalMaxPooling2D
import warnings
import pickle
warnings.filterwarnings('ignore')
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications import DenseNet121
import matplotlib.pyplot as plt
from sklearn.feature_selection import SelectKBest
from sklearn import preprocessing
from tensorflow.keras import Sequential


def inceptionModel(height, width, print_summary=False):
    model = InceptionV3(weights='imagenet', include_top=False, input_shape = (height, width, 3))

    print(model.summary())
    model.trainable = False
    output = GlobalMaxPooling2D()(model.outputs)
    model = Model(inputs=model.inputs, outputs=output)
    if print_summary:
        model.summary()
    return model

def getFeatureVector(model, image):
    featureVector = model.predict(image)
    featureVector = featureVector.flatten()
    return featureVector

In [5]:
file_path = 'dataset/with_mask/0-with-mask.jpg'
width = 224
height = 224
image = cv2.resize(cv2.imread(file_path), (width, height))
image = np.expand_dims(image, axis=0)

In [6]:
model = inceptionModel(width, height)
encoding = getFeatureVector(model, image)
encoding

87910968/87910968 [==============================] - 2s 0us/step
Model: "inception_v3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d (Conv2D)                (None, 111, 111, 32  864         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 batch_normalization (BatchNorm  (None, 111, 111, 32  96         ['conv2d[0][0]']                 
 alization)           

array([ 0.        , 15.98043   ,  0.        , ..., 13.049579  ,
        0.        ,  0.32117546], dtype=float32)

In [7]:
encoding.shape

(10240,)

In [8]:
encoding[:120]

array([  0.       ,  15.98043  ,   0.       ,  10.354565 ,  28.32877  ,
         2.8557606,   0.       ,  62.9658   ,   0.       ,  57.265305 ,
         0.8760957,   0.       ,   0.       ,  29.392052 ,   0.       ,
       143.26462  , 104.48839  ,   0.       ,  49.06828  ,   3.6876335,
         1.8175113,   0.       ,  59.213886 ,   3.7491984,  50.658337 ,
        10.600724 ,  23.697933 ,  16.615433 ,  25.795158 ,  21.309511 ,
         0.       ,   6.792663 ,  12.542621 ,  23.157957 ,   0.       ,
        31.299202 ,  35.182022 ,   0.       ,   9.016196 ,  26.399408 ,
         0.       ,   2.809493 ,   0.       ,   0.       ,  19.379463 ,
        37.62666  ,   0.       ,  27.05009  ,   0.       ,  24.89898  ,
         0.       ,  63.35244  ,  74.4602   ,  11.461222 ,  28.027384 ,
         0.       ,   6.90383  ,  44.08597  ,   0.       ,  73.908516 ,
        16.298199 , 125.69806  ,  18.036911 ,  50.268005 ,  53.019497 ,
         6.8312864,   2.9332078,  37.648087 ,  27.617182 ,  59.9